## Model Training

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [21]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

# Modelling
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, train_test_split

# Data Transformation
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# algorithms
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

# validation
from sklearn.model_selection import KFold, cross_validate

# others
import warnings

#### Import the CSV Data as Pandas DataFrame

In [2]:
df = pd.read_csv("data/StudentsPerformance.csv")

#### Show Top 5 Records

In [3]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


#### Preparing X and Y variables

In [4]:
X = df.drop(columns=['math score'],axis=1)

In [5]:
X.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [6]:
        
def get_categorical_columns (data:pd.DataFrame):
    """Return a list of column names that are categorical."""
    cat_cols = data.select_dtypes(include='object').columns
    return cat_cols

def get_numerical_columns(data:pd.DataFrame):
    """Return a list of column names that are numerical."""
    num_cols = data.select_dtypes(exclude='object').columns
    return num_cols

def get_categorical_columns_unique_values(data:pd.DataFrame):
    """Return a dictionary of columns and unique value
       dict = {'gender' : ['male','female'], 'race/ethnicity':[...],...}
    """
    cat_col_unique_vals = dict()
    cat_cols = get_categorical_columns(data=data)
    for col in cat_cols:
        unique_vals = data[col].unique().tolist()
        cat_col_unique_vals[col] = unique_vals
    return cat_col_unique_vals

    
def print_cat_cols_unique_vals(data:pd.DataFrame):
    """print the dict - {'gender' : ['male','female'], 'race/ethnicity':[...],...}"""
    cat_col_unique_vals = get_categorical_columns_unique_values(data=data)
    print("Categorical Columns : Unique Values in them")
    for cat in cat_col_unique_vals:
        print("Categories in ",cat," column :   ", cat_col_unique_vals[cat])
        
        
def get_categorical_columns_unique_value_dict(data:pd.DataFrame):
    cat_cols = get_categorical_columns(data=data)
    cat_summary = {col : list(data[col].value_counts().items()) for col in cat_cols}
    return cat_summary

In [7]:
print_cat_cols_unique_vals(data=df)

Categorical Columns : Unique Values in them
Categories in  gender  column :    ['female', 'male']
Categories in  race/ethnicity  column :    ['group B', 'group C', 'group A', 'group D', 'group E']
Categories in  parental level of education  column :    ["bachelor's degree", 'some college', "master's degree", "associate's degree", 'high school', 'some high school']
Categories in  lunch  column :    ['standard', 'free/reduced']
Categories in  test preparation course  column :    ['none', 'completed']


In [8]:
y = df['math score']

In [9]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math score, Length: 1000, dtype: int64

In [10]:
# Data Transformation :

# get the numeric and categoric columns
num_features = get_numerical_columns(data=X)
cat_features = get_categorical_columns(data=X)

# transformer objects
numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

# Column Transformer to apply transformatino seperately and then join
preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)


In [11]:
X = preprocessor.fit_transform(X)

In [12]:
X.shape

(1000, 19)

In [13]:
# separate dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((800, 19), (200, 19))

#### Create an Evaluate Function to give all metrics after model Training

In [15]:
# models algorithms dictionary, model list, r2 score list
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list =[]

In [ ]:
def print_scores(cv_train:dict[list],cv_val:dict[list],model_name:str):
        # CV Results with OVERFITTING CHECK
    if cv_train['cv_train_r2'] is not None:
        overfitting_gap = cv_train['cv_train_r2'] - cv_val['cv_r2']
        print(model_name)
        print(f"CV Train R2: {cv_train['cv_train_r2']:.4f} | CV Val R2: {cv_val['cv_r2']:.4f} (+/-{cv_val['cv_r2_std']:.4f})")
        print(f"CV Train RMSE: {cv_train['cv_train_rmse']:.4f}, CV Val RMSE: {cv_val['cv_rmse']:.4f}")
        print(f"CV Train MAE: {cv_train['cv_train_mae']:.4f}, CV Val MAE: {cv_val['cv_mae']:.4f}")
        print(f"Overfitting Gap: {overfitting_gap:.4f} | {'*** HIGH OVERFIT ***' if overfitting_gap > 0.1 else 'Good'}")
    
    model_list.append(model_name)
    r2_list.append(cv_val['cv_r2'])

In [13]:
def evaluate_model(true:np.ndarray,predicted:np.ndarray):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

NameError: name 'np' is not defined

In [ ]:
def get_model_evaluation(y_train:np.ndarray,y_test:np.ndarray,y_train_pred:np.ndarray,y_test_pred:np.ndarray,
                         model_name:str=None,cross_validation:bool=False):
    pass

In [ ]:
def get_nums():
    return ('mse','mae','rmse')

def get_eval():
    score_samp = ['mse','mae','rmse']
    train_samp = {}
    train_samp['mae'],train_samp[rmse']

IndentationError: expected an indented block after function definition on line 1 (783408571.py, line 1)

In [ ]:
def get_model_evaluation(y_train:np.ndarray,y_test:np.ndarray,y_train_pred:np.ndarray,y_test_pred:np.ndarray,
                         model_name:str,scores:list[str],cross_validation:bool=False)->tuple[dict,dict]:
    """
    Returns (train_dict,test_dict) : ({'mse':int_val,'rmse':init_val,..}, {..})
    """
    # train_scores = {model : {} for model in keys}
    
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
     

    """if cross_validation:
        return cv_train,cv_val"""
    
    """else:"""
    print(model_name)
    model_list.append(model_name)
     
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
        
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
        
    print('='*35)
    print('\n')

In [ ]:
def predict_models(X_train:np.ndarray,y_train:np.ndarray,X_test:np.ndarray,y_test:np.ndarray,models:dict,
                   scores:list[str]=['r2_score','rmse','mae'],cv_train:dict=None,cv_val:dict=None,cross_evaluation:bool=False):
    
    # initiate train and test scores/metrics
    # {train_score : {'mse':[...],'rmse':[...]}, test_score : {...}}
    result_scores = {
    'train_score': {model_name: {score: [] for score in scores} for model_name in models.keys()},
    'test_score': {model_name: {score: [] for score in scores} for model_name in models.keys()}
    }
    
    for model_name,model in models.items():
        # Train model
        model.fit(X_train, y_train) 

        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        """if cv_train:
            train_scores = cv_train
            test_scores = cv_test"""

        model_train_score, model_test_score = get_model_evaluation(y_train=y_train,y_test=y_test,y_train_pred=y_train_pred,
                                                                   y_test_pred=y_test_pred,model_name=model_name,scores=scores)

        # append the model scores/metrics
        for score_name in scores:
            result_scores['train_score'][model_name][score_name].append(model_train_score[score_name])
            result_scores['test_score'][model_name][score_name].append(model_test_score[score_name])
            
        # ....here
        if not cross_evaluation :
            print_scores(?model_name=model_name)

In [ ]:
def cross_validate_models(X:np.ndarray,y:np.ndarray,models:dict, n_splits:int=10):
    kf = KFold(n_splits=n_splits, shuffle=True, andom_state=42)
    
    # scores dict :
    cv_train = dict()
    cv_val = dict()
    
    for train_index, test_index in kf.split(X):
        X_train, X_test, y_train, y_test = X[train_index], X[test_index], y[train_index], y[test_index]
        predict_models(X_train=X_train,y_train=y_train,X_test=X_test,y_test=y_test,models=models,
                       cv_train=cv_train,cv_val=cv_val)

In [ ]:
# predict_models(X_train=X_train,y_train=y_train,X_test=X_test,y_test=y_test,models=models)

In [ ]:
def predict_models(X_train:np.ndarray, y_train:np.ndarray, X_test:np.ndarray, y_test:np.ndarray, models:dict):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for name, model in models.items():
        # CV with TRAIN SCORES enabled
        cv_results = cross_validate(model, X_train, y_train, cv=kf,
                                    scoring={'r2': 'r2',
                                             'mse': 'neg_mean_squared_error',
                                             'mae': 'neg_mean_absolute_error'},
                                    return_train_score=True)  # ← THIS IS KEY
        
        # Train scores (from CV training folds)
        cv_train_r2 = cv_results['train_r2'].mean()
        cv_train_rmse = np.sqrt(-cv_results['train_mse']).mean()
        
        # CV test scores (validation folds)
        cv_r2 = cv_results['test_r2'].mean()
        cv_r2_std = cv_results['test_r2'].std() * 2
        cv_rmse = np.sqrt(-cv_results['test_mse']).mean()
        cv_mae = -cv_results['test_mae'].mean()
        
        model.fit(X_train, y_train)
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        get_model_evaluation(y_train=y_train, y_test=y_test,
                             y_train_pred=y_train_pred, y_test_pred=y_test_pred,
                             name=name,
                             cv_train_r2=cv_train_r2, cv_r2=cv_r2, 
                             cv_rmse=cv_rmse, cv_mae=cv_mae, cv_r2_std=cv_r2_std)

In [ ]:
def get_model_evaluation(y_train:np.ndarray, y_test:np.ndarray, y_train_pred:np.ndarray, y_test_pred:np.ndarray,
                         name:str, cv_train_r2:float=None, cv_r2:float=None, cv_rmse:float=None, 
                         cv_mae:float=None, cv_r2_std:float=None):
    
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(name)
    model_list.append(name)
    
    # CV Results with OVERFITTING CHECK
    if cv_train_r2 is not None:
        overfitting_gap = cv_train_r2 - cv_r2
        print(f"CV Train R2: {cv_train_r2:.4f} | CV Val R2: {cv_r2:.4f} (+/-{cv_r2_std:.4f})")
        print(f"CV RMSE: {cv_rmse:.4f}, CV MAE: {cv_mae:.4f}")
        print(f"Overfitting Gap: {overfitting_gap:.4f} | {'*** HIGH OVERFIT ***' if overfitting_gap > 0.1 else 'Good'}")
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    # ... rest unchanged

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.3231
- Mean Absolute Error: 4.2667
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3940
- Mean Absolute Error: 4.2148
- R2 Score: 0.8804


Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5938
- Mean Absolute Error: 5.2063
- R2 Score: 0.8071
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.5197
- Mean Absolute Error: 5.1579
- R2 Score: 0.8253


Ridge
Model performance for Training set
- Root Mean Squared Error: 5.3233
- Mean Absolute Error: 4.2650
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3904
- Mean Absolute Error: 4.2111
- R2 Score: 0.8806


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 5.7077
- Mean Absolute Error: 4.5167
- R2 Score: 0.8555
-----------------------

In [20]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.3231
- Mean Absolute Error: 4.2667
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3940
- Mean Absolute Error: 4.2148
- R2 Score: 0.8804


Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5938
- Mean Absolute Error: 5.2063
- R2 Score: 0.8071
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.5197
- Mean Absolute Error: 5.1579
- R2 Score: 0.8253


Ridge
Model performance for Training set
- Root Mean Squared Error: 5.3233
- Mean Absolute Error: 4.2650
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3904
- Mean Absolute Error: 4.2111
- R2 Score: 0.8806


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 5.7077
- Mean Absolute Error: 4.5167
- R2 Score: 0.8555
-----------------------